In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.transforms import InterpolationMode, v2



In [2]:
# Repro + perf settings (keep close to original intent)
torch.manual_seed(3407)
if torch.cuda.is_available():
    torch.cuda.manual_seed(3407)

cudnn.deterministic = False
cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Paths (prefer competition canonical input path)
BASE = Path("/kaggle/input/cassava-leaf-disease-classification")
train_csv_path = BASE / "train.csv"
sample_sub_path = BASE / "sample_submission.csv"
train_dir = BASE / "train_images"
test_dir = BASE / "test_images"

assert train_csv_path.exists(), f"Missing: {train_csv_path}"
assert sample_sub_path.exists(), f"Missing: {sample_sub_path}"
assert train_dir.exists(), f"Missing: {train_dir}"
assert test_dir.exists(), f"Missing: {test_dir}"

# Basic params
img_size = 384
batch_size = 32
num_workers = min(4, os.cpu_count() or 1)
num_classes = 5
epochs = 2
lr = 3e-4




device: cuda


In [3]:
# Dataset that guarantees ordering matches a provided list (used for test submission alignment)
class CassavaImageDataset(Dataset):
    def __init__(self, img_dir: Path, image_ids, labels=None, transform=None):
        self.img_dir = Path(img_dir)
        self.image_ids = list(image_ids)
        self.labels = None if labels is None else list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = self.img_dir / image_id
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        if self.labels is None:
            return img, image_id
        return img, int(self.labels[idx])


# Transforms (keeps normalization same as original; adds resize for model input)
train_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

test_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)



In [4]:
# Load train + build a small train/val split for sanity (no effect on test submission format)
df = pd.read_csv(train_csv_path)
df["label"] = df["label"].astype(int)

# Deterministic split
rng = np.random.default_rng(3407)
idx = np.arange(len(df))
rng.shuffle(idx)
split = int(0.9 * len(df))
tr_idx, va_idx = idx[:split], idx[split:]

train_df = df.iloc[tr_idx].reset_index(drop=True)
val_df = df.iloc[va_idx].reset_index(drop=True)

train_ds = CassavaImageDataset(
    train_dir,
    train_df["image_id"].values,
    train_df["label"].values,
    transform=train_transforms,
)
val_ds = CassavaImageDataset(
    train_dir,
    val_df["image_id"].values,
    val_df["label"].values,
    transform=test_transforms,
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

# Test set must follow sample_submission ordering to avoid length/order issues
sample_sub = pd.read_csv(sample_sub_path)
test_image_ids = sample_sub["image_id"].tolist()
test_ds = CassavaImageDataset(
    test_dir, test_image_ids, labels=None, transform=test_transforms
)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,  # IMPORTANT: do not shuffle to preserve submission order
    num_workers=num_workers,
    pin_memory=True,
)

print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))



train/val/test sizes: 16848 1873 2676


In [5]:
# Model: replace missing external checkpoints with a standard torchvision backbone.
# This directly fixes FileNotFoundError/NameError while keeping a straightforward classifier approach.
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)



Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 112MB/s]


In [6]:
# Train briefly (kept short for runtime; no early stopping or approximations)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


for epoch in range(1, epochs + 1):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    n_tr = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = yb.size(0)
        tr_loss += loss.item() * bs
        tr_acc += accuracy_from_logits(logits.detach(), yb) * bs
        n_tr += bs

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    n_va = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)

            bs = yb.size(0)
            va_loss += loss.item() * bs
            va_acc += accuracy_from_logits(logits, yb) * bs
            n_va += bs

    print(
        f"epoch {epoch}/{epochs} | "
        f"train loss {tr_loss/n_tr:.4f} acc {tr_acc/n_tr:.4f} | "
        f"val loss {va_loss/n_va:.4f} acc {va_acc/n_va:.4f}"
    )



/tmp/ipykernel_55/2879490677.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_55/2879490677.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


epoch 1/2 | train loss 0.5175 acc 0.8203 | val loss 0.3955 acc 0.8553
epoch 2/2 | train loss 0.3643 acc 0.8757 | val loss 0.3541 acc 0.8756


In [7]:
# Inference on test in the exact sample_submission order and write a valid CSV
model.eval()
all_names = []
all_preds = []

with torch.no_grad():
    for xb, image_ids in test_loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        preds = logits.argmax(dim=1).detach().cpu().numpy().astype(int).tolist()
        all_names.extend(list(image_ids))
        all_preds.extend(preds)

# Validate submission length and ordering
assert len(all_names) == len(
    sample_sub
), f"Preds length {len(all_names)} != sample_submission {len(sample_sub)}"
# all_names should already equal sample_sub order since we constructed test_ds that way, but enforce safety:
pred_map = dict(zip(all_names, all_preds))
ordered_preds = [pred_map[iid] for iid in sample_sub["image_id"].tolist()]

submission = pd.DataFrame({"image_id": sample_sub["image_id"], "label": ordered_preds})
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with rows:", len(submission))

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
Wrote submission.csv with rows: 2676
